# Results Comparison Dashboard

This notebook loads all saved results JSON files produced by the model training notebooks
(`01_LSTM`, `02_CNN`, `03_CNN_LSTM`, `04_CNN_BiLSTM_Attention`, `05_Transformer`) and
produces a comprehensive comparative analysis.

**What it does:**
- Scans `results/` for all `*.json` result files
- Groups runs by model type and lets you select which run to compare
- Generates a summary table sorted by test accuracy
- Plots accuracy comparison, training curves, per-class F1 heatmap, and config comparison
- Identifies the best model across several dimensions
- Exports a Markdown comparison report

**Run all cells top-to-bottom** after at least one model notebook has been executed.

In [ ]:
# ── Cell 1: Install dependencies (safe to run even if already installed) ──
import subprocess, sys

required = ["pandas", "numpy", "matplotlib", "seaborn"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import os
import json
import glob
import warnings
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")

# Notebook-wide aesthetics
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

print("Imports OK")

In [ ]:
# ── Cell 2: CONFIG ──

RESULTS_DIR = 'results'   # Where model notebooks save their JSON files

# Model display order (used for consistent plot ordering)
MODEL_ORDER = ["lstm", "cnn", "cnn_lstm", "cnn_bilstm_attention", "transformer"]

# Palette: one colour per model (extended automatically if new models appear)
BASE_PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
                "#937860", "#DA8BC3", "#8C8C8C"]

# Run-selection policy: 'latest' | 'best_accuracy' | 'manual'
# When 'manual', set RUN_SELECTION dict below, e.g.
#   {"lstm": "20240601_120000", "cnn": "20240601_130000"}
RUN_SELECTION_POLICY = 'latest'
RUN_SELECTION_MANUAL: dict = {}   # only used when policy == 'manual'

In [ ]:
# ── Cell 3: Load all results ──

def load_all_results(results_dir: str) -> dict:
    """
    Scan results_dir for *.json files, load each, and group by 'model' key.
    Returns: {model_name: [result_dict, ...]} sorted by timestamp within each group.
    """
    pattern = os.path.join(results_dir, "*.json")
    files   = sorted(glob.glob(pattern))

    if not files:
        return {}

    grouped = defaultdict(list)
    for fpath in files:
        try:
            with open(fpath) as f:
                data = json.load(f)
            # Attach source filename for reference
            data["_source_file"] = fpath
            model_key = data.get("model", "unknown").lower().replace("-", "_")
            grouped[model_key].append(data)
        except Exception as e:
            print(f"  [WARN] Could not load {fpath}: {e}")

    # Sort each group by timestamp ascending
    for key in grouped:
        grouped[key].sort(key=lambda r: r.get("timestamp", ""))

    return dict(grouped)


def select_runs(grouped: dict, policy: str, manual: dict) -> dict:
    """
    Given grouped results, select one run per model according to policy.
    Returns: {model_name: result_dict}
    """
    selected = {}
    for model, runs in grouped.items():
        if not runs:
            continue

        if len(runs) == 1:
            selected[model] = runs[0]
            continue

        # Multiple runs — show available
        print(f"\nModel '{model}' — {len(runs)} run(s) found:")
        for i, r in enumerate(runs):
            ts  = r.get("timestamp", "unknown")
            acc = r.get("test_accuracy", float("nan"))
            print(f"  [{i}] timestamp={ts}  test_acc={acc:.4f}")

        if policy == "manual" and model in manual:
            target_ts = manual[model]
            match = [r for r in runs if r.get("timestamp") == target_ts]
            if match:
                chosen = match[0]
                print(f"  → Manual selection: {target_ts}")
            else:
                print(f"  [WARN] Timestamp '{target_ts}' not found; defaulting to latest.")
                chosen = runs[-1]
        elif policy == "best_accuracy":
            chosen = max(runs, key=lambda r: r.get("test_accuracy", 0.0))
            print(f"  → Best-accuracy selection: {chosen.get('timestamp')}")
        else:  # latest (default)
            chosen = runs[-1]
            print(f"  → Latest selection: {chosen.get('timestamp')}")

        selected[model] = chosen

    return selected


# ── Run loading ──
grouped_results = load_all_results(RESULTS_DIR)

if not grouped_results:
    print(
        "\n" + "="*60 + "\n"
        "No results found in the '" + RESULTS_DIR + "' directory.\n"
        "\nTo generate results, run any of the model training notebooks:\n"
        "  01_LSTM, 02_CNN, 03_CNN_LSTM,\n"
        "  04_CNN_BiLSTM_Attention, 05_Transformer\n"
        "\nEach notebook saves a JSON file to results/<model>_<timestamp>.json\n"
        "after training completes.\n"
        + "="*60
    )
    # Define empty sentinel so downstream cells fail gracefully
    SELECTED = {}
else:
    print(f"Found results for {len(grouped_results)} model(s): "
          f"{list(grouped_results.keys())}")
    SELECTED = select_runs(grouped_results, RUN_SELECTION_POLICY, RUN_SELECTION_MANUAL)
    print(f"\nSelected {len(SELECTED)} run(s) for comparison.")

In [ ]:
# ── Cell 4: Summary table ──

if not SELECTED:
    print("No results loaded — run a model notebook first.")
else:
    rows = []
    for model, r in SELECTED.items():
        cfg = r.get("config", {})

        # Epochs trained: length of loss history
        hist = r.get("history", {})
        epochs_trained = len(hist.get("loss", []))

        # Config summary fields (handle missing keys gracefully)
        config_summary = "|".join(filter(None, [
            f"filter={cfg['filter_type']}"         if "filter_type"   in cfg else None,
            f"norm={cfg['normalization']}"          if "normalization" in cfg else None,
            f"features={cfg['feature_count']}"      if "feature_count" in cfg else None,
            f"seq_len={cfg['sequence_length']}"     if "sequence_length" in cfg else None,
            f"batch={cfg['batch_size']}"            if "batch_size"    in cfg else None,
            f"lr={cfg['learning_rate']}"            if "learning_rate" in cfg else None,
        ])) or "N/A"

        rows.append({
            "Model":          model.upper().replace("_", " "),
            "Test Accuracy":  r.get("test_accuracy",  float("nan")),
            "Test Loss":      r.get("test_loss",       float("nan")),
            "Epochs Trained": epochs_trained if epochs_trained else "N/A",
            "Config Summary": config_summary,
            "Timestamp":      r.get("timestamp", "N/A"),
        })

    summary_df = (
        pd.DataFrame(rows)
        .sort_values("Test Accuracy", ascending=False)
        .reset_index(drop=True)
    )
    summary_df.index += 1  # 1-based rank
    summary_df.index.name = "Rank"

    # Pretty print
    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.float_format", "{:.4f}".format)
    display(summary_df)

In [ ]:
# ── Cell 5: Accuracy comparison bar chart ──

if not SELECTED:
    print("No results loaded — skipping plot.")
else:
    # Respect MODEL_ORDER; append any unknown models at the end
    ordered_models = [m for m in MODEL_ORDER if m in SELECTED] + \
                     [m for m in SELECTED   if m not in MODEL_ORDER]

    # Gather mean/std across ALL runs (not just selected), for error bars
    means, stds, labels_plot, colours = [], [], [], []
    colour_map = {m: BASE_PALETTE[i % len(BASE_PALETTE)]
                  for i, m in enumerate(ordered_models)}

    for m in ordered_models:
        all_accs = [r.get("test_accuracy", float("nan"))
                    for r in grouped_results.get(m, [])]
        all_accs = [a for a in all_accs if not np.isnan(a)]
        means.append(np.mean(all_accs) if all_accs else np.nan)
        stds.append(np.std(all_accs)   if len(all_accs) > 1 else 0.0)
        labels_plot.append(m.upper().replace("_", "\n"))
        colours.append(colour_map[m])

    fig, ax = plt.subplots(figsize=(max(6, len(ordered_models) * 1.6), 5))
    bars = ax.bar(
        labels_plot, means,
        yerr=stds if any(s > 0 for s in stds) else None,
        color=colours, width=0.55,
        error_kw=dict(ecolor="#333333", capsize=5, linewidth=1.5)
    )

    # Annotate bars
    for bar, val in zip(bars, means):
        if not np.isnan(val):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{val:.3f}",
                ha="center", va="bottom", fontsize=10, fontweight="bold"
            )

    ax.set_ylim(0, min(1.1, (max(m for m in means if not np.isnan(m)) + 0.12)
                      if any(not np.isnan(m) for m in means) else 1.1))
    ax.set_ylabel("Test Accuracy")
    ax.set_title("Model Test Accuracy Comparison\n"
                 "(error bars = std across multiple runs)", fontsize=13)
    ax.axhline(0.9, color="grey", linestyle="--", linewidth=0.8, alpha=0.6,
               label="90% threshold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "accuracy_comparison.png"),
                bbox_inches="tight")
    plt.show()
    print("Saved: results/accuracy_comparison.png")

In [ ]:
# ── Cell 6: Training curves comparison ──

if not SELECTED:
    print("No results loaded — skipping plot.")
else:
    ordered_models = [m for m in MODEL_ORDER if m in SELECTED] + \
                     [m for m in SELECTED   if m not in MODEL_ORDER]
    colour_map = {m: BASE_PALETTE[i % len(BASE_PALETTE)]
                  for i, m in enumerate(ordered_models)}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    fig.suptitle("Training Curves — All Models", fontsize=14, fontweight="bold")

    ax_loss, ax_acc = axes[0], axes[1]
    legend_handles = []

    for m in ordered_models:
        r    = SELECTED[m]
        hist = r.get("history", {})
        col  = colour_map[m]
        label = m.upper().replace("_", " ")

        # Loss
        if "loss" in hist and hist["loss"]:
            epochs = range(1, len(hist["loss"]) + 1)
            ax_loss.plot(epochs, hist["loss"],     color=col, linewidth=2,
                         label=f"{label} train")
            if "val_loss" in hist and hist["val_loss"]:
                ax_loss.plot(epochs, hist["val_loss"], color=col, linewidth=1.5,
                             linestyle="--")

        # Accuracy
        if "accuracy" in hist and hist["accuracy"]:
            epochs = range(1, len(hist["accuracy"]) + 1)
            line,  = ax_acc.plot(epochs, hist["accuracy"],     color=col,
                                 linewidth=2, label=f"{label} train")
            if "val_accuracy" in hist and hist["val_accuracy"]:
                ax_acc.plot(epochs, hist["val_accuracy"], color=col,
                            linewidth=1.5, linestyle="--")
            legend_handles.append(
                mpatches.Patch(color=col, label=label)
            )

    ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("Loss")
    ax_loss.set_title("Loss (solid=train, dashed=val)")
    ax_acc.set_xlabel("Epoch");  ax_acc.set_ylabel("Accuracy")
    ax_acc.set_title("Accuracy (solid=train, dashed=val)")
    ax_acc.set_ylim(bottom=0)

    if legend_handles:
        fig.legend(handles=legend_handles, loc="lower center",
                   ncol=min(len(legend_handles), 5),
                   bbox_to_anchor=(0.5, -0.05), fontsize=9)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(os.path.join(RESULTS_DIR, "training_curves.png"),
                bbox_inches="tight")
    plt.show()
    print("Saved: results/training_curves.png")

In [ ]:
# ── Cell 7: Per-class F1 comparison heatmap ──

if not SELECTED:
    print("No results loaded — skipping plot.")
else:
    # Gather F1 scores from classification reports
    f1_data = {}   # {model: {class_label: f1}}
    all_classes = set()

    for m, r in SELECTED.items():
        report = r.get("classification_report", {})
        if not report:
            continue
        class_f1 = {}
        for cls, metrics in report.items():
            # Skip aggregates
            if cls in ("accuracy", "macro avg", "weighted avg"):
                continue
            if isinstance(metrics, dict) and "f1-score" in metrics:
                class_f1[cls] = metrics["f1-score"]
                all_classes.add(cls)
        if class_f1:
            f1_data[m] = class_f1

    if not f1_data:
        print("No classification_report data found in loaded results.\n"
              "Ensure model notebooks save the full sklearn classification_report dict.")
    else:
        ordered_models = [m for m in MODEL_ORDER if m in f1_data] + \
                         [m for m in f1_data   if m not in MODEL_ORDER]
        sorted_classes = sorted(all_classes)

        # Build matrix: rows=classes, cols=models
        matrix = pd.DataFrame(
            index=sorted_classes,
            columns=[m.upper().replace("_", " ") for m in ordered_models],
            dtype=float
        )
        for m in ordered_models:
            col_name = m.upper().replace("_", " ")
            for cls in sorted_classes:
                matrix.loc[cls, col_name] = f1_data[m].get(cls, np.nan)

        fig_h = max(5, len(sorted_classes) * 0.4)
        fig_w = max(6, len(ordered_models) * 1.4)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))

        sns.heatmap(
            matrix.astype(float),
            ax=ax, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.0, vmax=1.0, linewidths=0.5, linecolor="#e0e0e0",
            cbar_kws={"label": "F1 Score"}
        )
        ax.set_title("Per-Class F1 Score by Model", fontsize=13, pad=12)
        ax.set_xlabel("Model", fontsize=11)
        ax.set_ylabel("Gesture Class", fontsize=11)
        ax.tick_params(axis="x", labelrotation=25)
        ax.tick_params(axis="y", labelrotation=0)

        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, "f1_heatmap.png"),
                    bbox_inches="tight")
        plt.show()
        print("Saved: results/f1_heatmap.png")

In [ ]:
# ── Cell 8: Config comparison table ──

if not SELECTED:
    print("No results loaded — skipping.")
else:
    # Collect all unique config keys across all models
    all_cfg_keys = set()
    for r in SELECTED.values():
        all_cfg_keys.update(r.get("config", {}).keys())
    all_cfg_keys = sorted(all_cfg_keys)

    if not all_cfg_keys:
        print("No 'config' data found in results JSON files.\n"
              "Ensure model notebooks save their CONFIG dict to the results JSON.")
    else:
        ordered_models = [m for m in MODEL_ORDER if m in SELECTED] + \
                         [m for m in SELECTED   if m not in MODEL_ORDER]
        col_names = [m.upper().replace("_", " ") for m in ordered_models]

        cfg_df = pd.DataFrame(index=all_cfg_keys, columns=col_names)
        for m, col in zip(ordered_models, col_names):
            cfg = SELECTED[m].get("config", {})
            for key in all_cfg_keys:
                val = cfg.get(key, "—")
                cfg_df.loc[key, col] = str(val)

        pd.set_option("display.max_colwidth", 40)
        print("Hyperparameter configuration comparison:")
        display(cfg_df)

In [ ]:
# ── Cell 9: Best model identification ──

if not SELECTED:
    print("No results loaded — skipping.")
else:
    print("=" * 60)
    print("  BEST MODEL SUMMARY")
    print("=" * 60)

    # ── 1. Best overall accuracy ──
    best_acc_model = max(
        SELECTED,
        key=lambda m: SELECTED[m].get("test_accuracy", 0.0)
    )
    best_acc_val = SELECTED[best_acc_model].get("test_accuracy", float("nan"))
    print(f"\n Best Test Accuracy : {best_acc_model.upper()} "
          f"({best_acc_val:.4f})")

    # ── 2. Best average F1 ──
    avg_f1_scores = {}
    for m, r in SELECTED.items():
        report = r.get("classification_report", {})
        if "weighted avg" in report and isinstance(report["weighted avg"], dict):
            avg_f1_scores[m] = report["weighted avg"].get("f1-score", float("nan"))
        elif "macro avg" in report and isinstance(report["macro avg"], dict):
            avg_f1_scores[m] = report["macro avg"].get("f1-score", float("nan"))

    if avg_f1_scores:
        best_f1_model = max(avg_f1_scores, key=avg_f1_scores.get)
        print(f" Best Avg F1 Score  : {best_f1_model.upper()} "
              f"({avg_f1_scores[best_f1_model]:.4f})")
    else:
        print(" Best Avg F1 Score  : N/A (no classification_report found)")

    # ── 3. Fastest convergence (fewest epochs to reach best val_accuracy) ──
    conv_epochs = {}
    for m, r in SELECTED.items():
        hist    = r.get("history", {})
        val_acc = hist.get("val_accuracy", [])
        if val_acc:
            best_val = max(val_acc)
            epoch    = val_acc.index(best_val) + 1   # 1-based
            conv_epochs[m] = epoch

    if conv_epochs:
        fastest_model = min(conv_epochs, key=conv_epochs.get)
        print(f" Fastest Convergence: {fastest_model.upper()} "
              f"(best val_acc at epoch {conv_epochs[fastest_model]})")
        for m, ep in sorted(conv_epochs.items(), key=lambda x: x[1]):
            print(f"   {m.upper():<30} → epoch {ep}")
    else:
        print(" Fastest Convergence: N/A (no val_accuracy history found)")

    # ── 4. Lightest model (param count) ──
    param_counts = {}
    for m, r in SELECTED.items():
        cfg = r.get("config", {})
        pc  = cfg.get("param_count", cfg.get("total_params", None))
        if pc is not None:
            try:
                param_counts[m] = int(pc)
            except (ValueError, TypeError):
                pass

    if param_counts:
        lightest = min(param_counts, key=param_counts.get)
        print(f"\n Lightest Model     : {lightest.upper()} "
              f"({param_counts[lightest]:,} params)")
        for m, pc in sorted(param_counts.items(), key=lambda x: x[1]):
            print(f"   {m.upper():<30} → {pc:,} params")
    else:
        print("\n Lightest Model     : N/A "
              "(save param_count in config dict to enable this)")

    print("\n" + "=" * 60)

In [ ]:
# ── Cell 10: Export comparison report (Markdown) ──

if not SELECTED:
    print("No results loaded — nothing to export.")
else:
    ts_now   = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join(RESULTS_DIR, f"comparison_report_{ts_now}.md")

    lines = []
    lines.append(f"# Results Comparison Report")
    lines.append(f"")
    lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"")
    lines.append(f"## Summary Table")
    lines.append(f"")

    # Summary table as markdown
    ordered_models = [m for m in MODEL_ORDER if m in SELECTED] + \
                     [m for m in SELECTED   if m not in MODEL_ORDER]

    # Header
    lines.append("| Rank | Model | Test Accuracy | Test Loss | Epochs | Timestamp |")
    lines.append("|------|-------|--------------|-----------|--------|-----------|")

    sorted_models = sorted(
        ordered_models,
        key=lambda m: SELECTED[m].get("test_accuracy", 0.0),
        reverse=True
    )
    for rank, m in enumerate(sorted_models, 1):
        r       = SELECTED[m]
        acc     = r.get("test_accuracy", float("nan"))
        loss    = r.get("test_loss",      float("nan"))
        hist    = r.get("history", {})
        epochs  = len(hist.get("loss", []))
        ts      = r.get("timestamp", "N/A")
        lines.append(
            f"| {rank} | {m.upper().replace('_',' ')} "
            f"| {acc:.4f} | {loss:.4f} | {epochs} | {ts} |"
        )

    lines.append("")
    lines.append("## Best Model Analysis")
    lines.append("")

    # Best accuracy
    best_m = sorted_models[0]
    lines.append(f"- **Best Test Accuracy**: `{best_m.upper()}` — "
                 f"{SELECTED[best_m].get('test_accuracy', float('nan')):.4f}")

    # Best F1
    if avg_f1_scores:
        best_f1_m = max(avg_f1_scores, key=avg_f1_scores.get)
        lines.append(f"- **Best Avg F1**: `{best_f1_m.upper()}` — "
                     f"{avg_f1_scores[best_f1_m]:.4f}")

    # Fastest convergence
    if conv_epochs:
        fastest = min(conv_epochs, key=conv_epochs.get)
        lines.append(f"- **Fastest Convergence**: `{fastest.upper()}` — "
                     f"reached best val_acc at epoch {conv_epochs[fastest]}")

    # Lightest
    if param_counts:
        lightest = min(param_counts, key=param_counts.get)
        lines.append(f"- **Lightest Model**: `{lightest.upper()}` — "
                     f"{param_counts[lightest]:,} parameters")

    lines.append("")
    lines.append("## Config Comparison")
    lines.append("")

    # Config table
    all_cfg_keys = sorted(set().union(*[
        r.get("config", {}).keys() for r in SELECTED.values()
    ]))
    if all_cfg_keys:
        header_cols = [m.upper().replace("_", " ") for m in sorted_models]
        lines.append("| Config Key | " + " | ".join(header_cols) + " |")
        lines.append("|".join(["---"] * (len(header_cols) + 1)) + "|")
        for key in all_cfg_keys:
            row_vals = [
                str(SELECTED[m].get("config", {}).get(key, "—"))
                for m in sorted_models
            ]
            lines.append(f"| `{key}` | " + " | ".join(row_vals) + " |")

    lines.append("")
    lines.append("## Convergence Summary")
    lines.append("")
    if conv_epochs:
        lines.append("| Model | Epoch of Best Val Accuracy |")
        lines.append("|-------|---------------------------|")
        for m, ep in sorted(conv_epochs.items(), key=lambda x: x[1]):
            lines.append(f"| {m.upper().replace('_', ' ')} | {ep} |")
    else:
        lines.append("_No val_accuracy history available._")

    lines.append("")
    lines.append("---")
    lines.append("*Auto-generated by 00_Results_Comparison.ipynb*")

    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(out_path, "w") as f:
        f.write("\n".join(lines))

    print(f"Report saved to: {out_path}")